In [ ]:
import torch
import torch.nn.functional as F
# Efficient implementation equivalent to the following:
# query: [B,H,L,D]
# key: [B,H,S,D]

def scaled_dot_product_attention(query, key, value, attn_mask=None, dropout_p=0.0,
        is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:
    L, S = query.size(-2), key.size(-2)
    scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
    attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
    if is_causal:
        assert attn_mask is None
        temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
        attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))

    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_bias.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attn_bias = attn_mask + attn_bias

    if enable_gqa:
        key = key.repeat_interleave(query.size(-3)//key.size(-3), -3)
        value = value.repeat_interleave(query.size(-3)//value.size(-3), -3)

    attn_weight = query @ key.transpose(-2, -1) * scale_factor
    attn_weight += attn_bias
    attn_weight = torch.softmax(attn_weight, dim=-1)
    attn_weight = torch.dropout(attn_weight, dropout_p, train=True)
    return attn_weight @ value

In [7]:
# Optionally use the context manager to ensure one of the fused kernels is run
query = torch.rand(32, 8, 128, 64, dtype=torch.float16, device="cpu")
key = torch.rand(32, 8, 128, 64, dtype=torch.float16, device="cpu")
value = torch.rand(32, 8, 128, 64, dtype=torch.float16, device="cpu")
F.scaled_dot_product_attention(query,key,value)


tensor([[[[0.5181, 0.5010, 0.4651,  ..., 0.4966, 0.5396, 0.4895],
          [0.5210, 0.4929, 0.4641,  ..., 0.4995, 0.5396, 0.4822],
          [0.5200, 0.4993, 0.4634,  ..., 0.4949, 0.5371, 0.4883],
          ...,
          [0.5171, 0.5020, 0.4636,  ..., 0.4985, 0.5396, 0.4878],
          [0.5161, 0.4978, 0.4644,  ..., 0.4978, 0.5381, 0.4885],
          [0.5151, 0.4932, 0.4663,  ..., 0.5024, 0.5405, 0.4895]],

         [[0.5176, 0.5137, 0.4846,  ..., 0.5225, 0.4968, 0.4797],
          [0.5220, 0.5166, 0.4829,  ..., 0.5195, 0.4968, 0.4797],
          [0.5176, 0.5205, 0.4841,  ..., 0.5205, 0.4958, 0.4753],
          ...,
          [0.5132, 0.5171, 0.4788,  ..., 0.5210, 0.4932, 0.4792],
          [0.5200, 0.5171, 0.4810,  ..., 0.5225, 0.4934, 0.4802],
          [0.5225, 0.5176, 0.4768,  ..., 0.5225, 0.4922, 0.4797]],

         [[0.5044, 0.5010, 0.5190,  ..., 0.5337, 0.4883, 0.4785],
          [0.5063, 0.5015, 0.5244,  ..., 0.5381, 0.4873, 0.4780],
          [0.5063, 0.5010, 0.5269,  ..., 0

In [ ]:
def scaled_dot_product_attention(query, key, value, attn_mask=None, dropout_p=0.0,
        is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:
        #query [B,H,L,D]
        #key [B,H,S,D]
        #value [B,H,S,Dv]
        L = query.size(-2)
        S = key.size(-2)
        d = query.size(-1)
        scale_factor = 1 / sqrt(d) if scale is None else scale
        attention_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
        if is_causal ==True:
                assert attn_mask == None
                temp_mask = torch.ones(L,S,dtype=torch.bool).tril(diagonal=0)
                attention_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
        attention_weight = query @ key.transpose(-2,-1) * scale_factor
        attention_weight += attention_bias
        attention_weight = torch.softmax(attention_weight, dim=-1)
        #第一次写忘记过激活函数了，也忘了乘以scaler
        results = attention_weight @ value


        

In [ ]:
from torch import Tensor
def scaled_dot_product_attention(
    query:Tensor, #[B，H，L，D]
    key: Tensor, #[B，H，S，D]
    value: Tensor,  # [B, H, S, Dv]
    attn_mask: Optional[Tensor] = None,  # [L, S] 
) -> Tensor:
    L = query.size(-2)
    S = key.size(-2)
    scale_factor = 1 / math.sqrt(query.size(-1))
    attention_weight = query @ key.transpose(-2,-1) *scale_factor
    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attention_weight.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attention_weight = attention_weight + attn_mask
    attention_weight = torch.softmax(attention_weight,dim=-1)
    return attention_weight @ value